In [1]:
from pathlib import Path
import re

___
### Extracting useful info from log file

Log file contains lot of metadata, such as date, time, process id at the beginning of each line. This is unnecessary for our data<br><br>
We only need lines with block id information, and the message in the line to build template<br><br>
Example line from the log:<br>
081111 041138 19464 INFO dfs.DataNodeDataXceiver: Receiving block blk_-4750397141673817648 src: /10.251.70.112:52693 dest: /10.251.70.112:50010<br><br>
Here, "081111 041138 19464 INFO dfs.DataNodeDataXceiver" is unnecessary. We only need "blk_-4750397141673817648", and "Receiving block blk_-4750397141673817648 src:(.) dest: (.) <br><br>

Regex for block id: (blk_-?\d+)<br>
&ensp;blk_  -  must be present<br>
&ensp;-?    -  '-' is optional<br>
&ensp;\d+   -   any number of digits after that is acceptable<br><br>

Parsing message:<br>
From sample line, we understand that message part lies after the first colon. Hence, split line based on first colon, then second part of sentence will be message. Message still not cleaned properly at this stage, as things like src and dest will still have numbers

In [2]:
log_path = Path('../datasets/HDFS_v1/HDFS.log')

def find_pattern(log_path): 
    data = []
    
    blk_pattern = re.compile(r'(blk_-?\d+)')

    with open(log_path) as f:
        for line in f:
            blk_match = blk_pattern.search(line)
            if not blk_match:
                continue

            blk_id = blk_match.group(1)

            parts = line.split(':', maxsplit=1)
            if len(parts) < 2:
                continue
            msg = parts[1].strip()

            data.append((blk_id, msg))

    return data

In [3]:
parsed_data = find_pattern(log_path)
print('No of lines parsed: ', len(parsed_data))

No of lines parsed:  11175629


In [4]:
import numpy as np
print('5 sample lines: ')
for i in range(5):
    rand = np.random.randint(0, len(parsed_data))
    print(parsed_data[rand])

5 sample lines: 
('blk_-3379373600084985901', 'Deleting block blk_-3379373600084985901 file /mnt/hadoop/dfs/data/current/subdir33/blk_-3379373600084985901')
('blk_9004035938914413957', 'Received block blk_9004035938914413957 of size 67108864 from /10.250.10.144')
('blk_-4037345778299640341', 'BLOCK* NameSystem.delete: blk_-4037345778299640341 is added to invalidSet of 10.251.42.9:50010')
('blk_-6571371446950308980', 'BLOCK* NameSystem.allocateBlock: /user/root/randtxt2/_temporary/_task_200811101024_0002_m_001761_0/part-01761. blk_-6571371446950308980')
('blk_-7421183144028569819', 'BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.70.5:50010 is added to blk_-7421183144028569819 size 67108864')


___

### Normalising message to follow template structure
<br>
Messages(events) have been extracted, but they contain blk_id, source and destination IPs, size etc, which vary from line to line, even if the event is the same. This would cause the each line to be classfied as different event<br>
Hence, next step:<br><br>

Normalise event messages<br>
Ex: Convert 'Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010' &ensp; TO &ensp; 'Receiving block (.) src: (.) dest: (.)'<br><br>

Regex used:<br>
&ensp; subbing "blk_-?\d+" ----- Subbing blk_id<br>
&ensp; subbing "/?\d+\.\d+\.\d+\.\d+(?::\d+)?:?" ----- Subbing IPs and ports<br>
&ensp; subbing "\b\d+\b" ----- Subbing standalone numbers such as size<br><br>

Noticed that one more regex subbing has to be applied for treating directories, like in cases of:<br><br>
('blk_-3100963327775157803', 'Deleting block <> file /mnt/hadoop/dfs/data/current/subdir16/<>')<br>
('blk_7185070990871707682', 'Deleting block <> file /mnt/hadoop/dfs/data/current/subdir50/<>')<br>
The paths in these two examples will cause the two to be separate templates, although they are evidently performing the same operation<br><br>

Solution for paths:<br>
&ensp; subbing "/[^\s]+" ---- removes anything that starts with a / (mostly paths)

In [5]:
def normalise_message(msg):
    msg = re.sub(r"blk_-?\d+", "<*>", msg) # Removing blk_id
    msg = re.sub(r"/?\d+\.\d+\.\d+\.\d+(?::\d+)?:?", "<*>", msg) # Removing IP addresses and ports
    msg = re.sub(r"\b\d+\b", "<*>", msg) # Removing standalone numbers like size
    msg = re.sub(r"/[^\s]+", "<*>", msg) # Removing paths 

    return msg

In [6]:
# Im saving under same variable name to save memory
parsed_data = [
    (blk_id, normalise_message(msg)) for blk_id, msg in parsed_data
]

In [7]:
print('5 Random normalised data: ')
for i in range(5):
    rand = np.random.randint(0, len(parsed_data))
    print(parsed_data[rand])

5 Random normalised data: 
('blk_-8200112767031306196', 'Receiving block <*> src: <*> dest: <*>')
('blk_-1511208889345843978', 'BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>')
('blk_-8912301699117206900', 'Receiving block <*> src: <*> dest: <*>')
('blk_-7525496269943909625', 'Deleting block <*> file <*>')
('blk_-1396210182605857686', 'Received block <*> of size <*> from <*>')


___
### Encoding normalised messages into Event IDs
<br>
We now have normalised messages, ie, same events have same message structure, and all the variables like IPs and blk_ids have been removed<br>
Next step is to assign IDs to these messages, so that they can be identified with numbers instead of strings, as to feed them to the model<br><br>

Goal:<br>
Convert (blk_id, normalised_msg) -> (blk_id, event_id)<br>

This way, we can later group same blk_ids and put them into sequences. LSTM will not understand strings, but will be able to understand numbers, which is why normalised messages are being given numerical IDs<br><br>

How it's done:<br>
Use event_dict to map normalised messages to a particular ID. Return normalised_data with ID of normalised msg, and event_dict to know what ID is what event, and number of unique events

In [8]:
def build_event_mapping(normalised_data):
    encoded_data = []
    event_dict = {}
    event_id = 0

    for blk_id, msg in normalised_data:
        if msg not in event_dict:
            event_dict[msg] = event_id
            event_id += 1

        encoded_data.append((blk_id, event_dict[msg]))

    return encoded_data, event_dict

In [9]:
encoded_data, event_dict = build_event_mapping(parsed_data)

print('No of unique events: ', len(event_dict))
print('5 random samples of encoded data: ')
for i in range(25):
    rand = np.random.randint(0, len(encoded_data))
    print(encoded_data[rand])

No of unique events:  54
5 random samples of encoded data: 
('blk_6602285322127165799', 4)
('blk_1918814142745188757', 9)
('blk_6615975681551118991', 1)
('blk_-7735129651345870762', 2)
('blk_4090314397308210046', 2)
('blk_950962581134282809', 0)
('blk_-8306635528620904064', 16)
('blk_127553258816787494', 3)
('blk_6780067540910367912', 16)
('blk_6791283604927267356', 3)
('blk_-4769148503783467914', 4)
('blk_-4013488363967729201', 33)
('blk_-1365898484604170304', 33)
('blk_5708722897243745454', 2)
('blk_9101441027185481223', 16)
('blk_1384854280246503534', 0)
('blk_-5345004875932382351', 2)
('blk_8756953578757192281', 4)
('blk_-3626278050629767403', 4)
('blk_-4118221860884892618', 16)
('blk_-7926534347049606609', 36)
('blk_-151037359366940037', 0)
('blk_8249266768963607435', 16)
('blk_6357234316513487185', 0)
('blk_8878221659550636491', 3)


In [10]:
event_dict

{'Receiving block <*> src: <*> dest: <*>': 0,
 'BLOCK* NameSystem.allocateBlock: <*> <*>': 1,
 'PacketResponder <*> for block <*> terminating': 2,
 'Received block <*> of size <*> from <*>': 3,
 'BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>': 4,
 'Received block <*> src: <*> dest: <*> of size <*>': 5,
 '<*>Transmitted block <*> to <*>': 6,
 '<*> Starting thread to transfer block <*> to <*>, <*>': 7,
 'BLOCK* ask <*> to replicate <*> to datanode(s) <*> <*>': 8,
 '<*> Served block <*> to <*>': 9,
 'BLOCK* ask <*> to replicate <*> to datanode(s) <*>': 10,
 '<*> Starting thread to transfer block <*> to <*>': 11,
 'Verification succeeded for <*>': 12,
 'writeBlock <*> received exception java.net.SocketTimeoutException': 13,
 'PacketResponder <*> <*> Exception java.io.EOFException': 14,
 'writeBlock <*> received exception java.io.IOException: Could not read from stream': 15,
 'Deleting block <*> file <*>': 16,
 'Receiving empty packet for block <*>': 17,
 

#### More events according to my function (54) compared to preprocessed ones(23 or 24). I still don't think this is a problem, since certain similar events are indeed different, such as<br>
 'writeBlock <> received exception java.net.NoRouteToHostException: No route to host': 49,<br>
 'writeBlock <> received exception java.io.IOException: Broken pipe': 50,<br>
 'writeBlock <> received exception java.io.IOException: Interrupted receiveBlock': 47,<br><br>

Some are almost exactly similar, but I am not cleaning these, as the model will be able to treat them similarly, and it's still only 54 events, and not 1000s, and even the very similar messages could still carry different meanings. Unless 100% sure, I cannot write off the possibility of them having different meanings, hence I will keep the events as they are. If they are very similar, model will learn to weigh them accordingly anyway. For me, the risk of losing potentially meaningful signal is not worth very slight optimization
___

___
### Grouping all events with same blk_id
Currently, we have millions of events, but they are not put together. Hence, next step is to put all events of the same blk_id into a single sequence. These sequences will be individual examples/rows for training and testing<br><br>

Goal: Convert:<br>
('blk_1', 3)<br>
('blk_2', 7)<br>
('blk_1', 5)<br>
('blk_1', 2)<br>
('blk_2', 1) &ensp;&ensp;&ensp; TO:<br><br>
blk_1 → [3, 5, 2]<br>
blk_2 → [7, 1<br><br>

How it's done:<br>
Again, use a dictonary, where keys are blk_ids and values are lists of event_ids in sequence

In [11]:
from collections import defaultdict

def build_sequence(encoded_data):
    sequences = defaultdict(list)

    for blk, event_id in encoded_data:
        sequences[blk].append(event_id)

    return sequences

In [12]:
sequences = build_sequence(encoded_data)

In [13]:
print('No of blocks: ', len(sequences))
print('5 random samples from sequences:')
for i, (blk, events) in enumerate(sequences.items()):
    print(blk, events)
    print()
    if i == 4:
        break

No of blocks:  575061
5 random samples from sequences:
blk_-1608999687919862906 [0, 1, 0, 0, 2, 2, 3, 3, 2, 3, 4, 4, 4, 5, 0, 6, 5, 0, 7, 8, 4, 4, 9, 8, 5, 5, 0, 0, 6, 7, 4, 4, 0, 5, 0, 6, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 8, 5, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 10, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 11, 5, 0, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 33, 33, 33, 33, 33, 33, 33, 33, 33, 33, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16]

blk_7503483334202473044 [0, 0, 1, 0, 2, 3, 2, 3, 2, 3, 4, 4, 4, 9, 12, 12, 33, 33, 33, 16

#### Things seem right. Some blocks have more number of events, some have small. Different events are happening in each block etc
___

___
### Attaching labels to sequences
Sequences belonging to a block have been created. Now, using anomaly_labels.csv, I attach labels to each block. I will create a dataframe of blk_id, events and labels.<br><br>

Goal: <br>
Make a dataframe with blk_id, sequences and labels. sequences can then be used as X, and labels can be used as y. blk_id will be useful when receiving user input<br><br>

How it's done:<br>
1. Create dataframe from sequences: df = pd.DataFrame(sequences.items(), columns=['blk_id', 'events_sequence'])
2. Create dataframe from labels csv: labels_df = pd.read_csv('../datasets/HDFS_v1/anomaly_label.csv')
3. For merging two dataframes by blk_id, they have to have the same column name, hence rename in labels_df: labels_df.rename(columns={'BlockId': 'blk_id'}, inplace=True)
4. Merge: df = df.merge(labels_df, on='blk_id')
5. Map 0 to Normal and 1 to Anomaly: df['Label'] = df['Label'].map({'Normal': 0, 'Anomaly': 1})

In [14]:
import pandas as pd

In [33]:
df = pd.DataFrame(sequences.items(), columns=['blk_id', 'events_sequence'])
df.sample(5)

,blk_id,events_sequence
212455,blk_4155864903556153987,"[0, 0, 0, 1, 2, 3, 2, 3, 2, 3, 4, 4, 20, 4, 33..."
463676,blk_5236485828350107893,"[0, 0, 0, 1, 2, 3, 2, 3, 4, 4, 2, 3, 4, 33, 33..."
271341,blk_-8491308539174894521,"[0, 0, 0, 1, 2, 3, 2, 3, 4, 4, 2, 3, 4, 33, 33..."
113081,blk_-7714844530540164671,"[1, 0, 0, 0, 4, 2, 3, 4, 4, 2, 3, 2, 3, 36, 9,..."
370947,blk_-3019561281133080838,"[0, 0, 1, 0, 2, 3, 2, 3, 2, 3, 4, 4, 11, 10, 4..."


In [34]:
labels_df = pd.read_csv('../datasets/HDFS_v1/anomaly_label.csv')
labels_df.rename(columns={'BlockId': 'blk_id'}, inplace=True)
labels_df.sample(3)

,blk_id,Label
339447,blk_3871083650741470626,Normal
192065,blk_-7727359630283175124,Normal
412255,blk_-3913549962648455636,Normal


In [35]:
df = df.merge(labels_df, on='blk_id')
df.sample(3)

,blk_id,events_sequence,Label
269536,blk_4155626601702183401,"[0, 0, 0, 1, 2, 3, 2, 3, 4, 4, 2, 3, 4, 33, 33...",Normal
390290,blk_2565909344225184419,"[0, 0, 0, 1, 2, 3, 2, 3, 2, 3, 4, 4, 4, 33, 33...",Normal
572206,blk_2031734918591304637,"[0, 0, 0, 1, 2, 3, 3, 4, 4, 2, 2, 3, 4, 33, 33...",Normal


In [36]:
df['Label'] = df['Label'].map({'Normal': 0, 'Anomaly': 1})

In [45]:
df.sample(3)

,blk_id,events_sequence,Label
154348,blk_-6538891058531862642,"[0, 1, 0, 0, 2, 3, 2, 3, 4, 4, 2, 3, 4, 9, 9, ...",1
254271,blk_4397075619947553714,"[0, 0, 0, 1, 2, 3, 2, 3, 2, 3, 4, 4, 4, 12, 33...",0
466713,blk_738634981694991499,"[0, 0, 0, 1, 2, 3, 2, 3, 2, 3, 4, 4, 4, 33, 33...",0


In [46]:
print('No of blocks: ', len(df))

No of blocks:  575061


Matches correctly with length of sequences. Exporting df as csv for EDA 

In [51]:
df.to_csv('../datasets/HDFS_v1/prepared_df.csv', index=False)

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 575061 entries, 0 to 575060
Data columns (total 3 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   blk_id           575061 non-null  str   
 1   events_sequence  575061 non-null  object
 2   Label            575061 non-null  int64 
dtypes: int64(1), object(1), str(1)
memory usage: 13.2+ MB
